# Expansion Contraction
Configuration: `stages/03_expansion_contraction/config/runtime.json`.


### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
expansion_contraction_experiment_dir = ROOT / 'experiment/research/trapping_sets/stages/03_expansion_contraction'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

expansion_contraction_data_paths = ExperimentPaths(ROOT, expansion_contraction_experiment_dir / 'config/paths.json')

import json

expansion_contraction_config = json.loads((expansion_contraction_experiment_dir / 'config/runtime.json').read_text())
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.

from tqdm.auto import tqdm
from experiment.utils.trapping_sets import save_plotly

expansion_contraction_plots = expansion_contraction_data_paths.plots()
expansion_contraction_plots.mkdir(parents=True, exist_ok=True)

from experiment.utils.trapping_sets import parse_node_ids


In [ ]:
import sys
import os
from pathlib import Path
import itertools
from dataclasses import dataclass, replace
from typing import Callable, Iterable, Optional, Tuple, Dict, Set, List, Any
import stim
import numpy as np
import networkx as nx
from beliefmatching import detector_error_model_to_check_matrices

expansion_contraction_data_dir = expansion_contraction_data_paths.run()
expansion_contraction_data_dir.mkdir(parents=True, exist_ok=True)

from utils.misc import *
from utils.graph_struct import build_tanner_graph_from_dem, local_incident_subgraph
from utils.graph_vis import visualize_graph_plotly, visualize_det_on_tanner_graph, flatten_graph

%load_ext autoreload
%autoreload 2


In [ ]:
from codes.ca_code import CA_5_2_3, CA_9_2_5
from src.noises_ca import CircuitBuilder, ErrorRate
from src.decoders import BPOSD as expansion_contraction_bposd
from src.decoders import Tesseract


In [ ]:
expansion_contraction_ca_codes = {'CA_5_2_3': CA_5_2_3(), 'CA_9_2_5': CA_9_2_5()}

from experiment.utils import decoder_setup

def build_decoder_pair(code, p):
    return decoder_setup.build_decoder_pair(code, p, bposd_type=expansion_contraction_bposd, tesseract_type=Tesseract)


# Subgraph Expansion & Contraction

cycles -> trapping set

### Define and inspect local constructions

Prepare the expansion/contraction record helpers and reconstruct the selected DEM. The following exploratory examples connect local structures with explicit decoder inputs.


In [ ]:
@dataclass
class TrappingSetRecord:
    variables: Tuple[int, ...]
    checks: Tuple[int, ...]
    odd_checks: Tuple[int, ...]
    a: int
    b: int
    critical_number: int
    strength: int
    minimal_failures: Tuple[Tuple[int, ...], ...]
    seed_cycle: Tuple[Any, ...]
    expanded_variables: Tuple[int, ...]
    expanded_checks: Tuple[int, ...]

    def to_dict(self) -> Dict[str, Any]:
        return {
            'variables': self.variables,
            'checks': self.checks,
            'odd_checks': self.odd_checks,
            'a': self.a,
            'b': self.b,
            'critical_number': self.critical_number,
            'strength': self.strength,
            'minimal_failures': self.minimal_failures,
            'seed_cycle': self.seed_cycle,
            'expanded_variables': self.expanded_variables,
            'expanded_checks': self.expanded_checks,
        }

from experiment.utils import bp_message_dynamics

Matrices = bp_message_dynamics.Matrices
DecoderFails = Callable[[Matrices, np.ndarray], bool]
bposd, tess = build_decoder_pair(expansion_contraction_ca_codes['CA_9_2_5'], p=0.01)

def bposd_fails_but_tesseract_succeeds(matrices: Matrices, e_sub):
    dem = matrices.to_dem()
    obs = matrices.L @ e_sub % 2
    syndrome = matrices.H @ e_sub % 2
    compiled_b = bposd.compile_decoder_for_dem(dem=dem)
    compiled_t = tess.compile_decoder_for_dem(dem=dem)
    pred_b = compiled_b.decoder.decode(syndrome)
    pred_t = compiled_t.decoder.decode(syndrome)
    b_succ = np.array_equal(pred_b, obs)
    t_succ = np.array_equal(pred_t, obs)

    return not b_succ and t_succ

def bposd_fails(matrices: Matrices, e_sub):
    dem = matrices.to_dem()
    obs = matrices.L @ e_sub % 2
    syndrome = matrices.H @ e_sub % 2
    compiled_b = bposd.compile_decoder_for_dem(dem=dem)
    pred_b = compiled_b.decoder.decode(syndrome)
    b_succ = np.array_equal(pred_b, obs)

    return not b_succ

def build_tanner_graph(H: np.ndarray) -> nx.Graph:
    """
    Build Tanner graph from parity-check matrix H.

    Variable nodes are ("v", j)
    Check nodes are ("c", i)
    """

    m, n = H.shape
    G = nx.Graph()
    G.add_nodes_from((('v', j) for j in range(n)))
    G.add_nodes_from((('c', i) for i in range(m)))
    rows, cols = H.nonzero()

    for i, j in tqdm(zip(rows, cols), desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        G.add_edge(('c', int(i)), ('v', int(j)))

    return G

def canonical_cycle(cycle: Iterable[Any]) -> Tuple[Any, ...]:
    """
    Canonicalize an undirected cycle up to rotation and reversal.
    cycle should not repeat the first node at the end.
    """

    c = list(cycle)
    reps = []
    for seq in tqdm((c, list(reversed(c))), desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        for k in tqdm(range(len(seq)), desc='Trapping-set analysis', leave=False, disable=expansion_contraction_config['print_level'] == 1):
            reps.append(tuple(seq[k:] + seq[:k]))

    return min(reps, key=lambda x: tuple(map(str, x)))

def enumerate_short_cycles(
    G: nx.Graph,
    max_len: int=8,
    max_cycles: Optional[int]=None,
) -> List[Tuple[Any, ...]]:
    """
    Enumerate simple cycles up to max_len in an undirected Tanner graph.
    """

    nodes = sorted(G.nodes(), key=str)
    rank = {u: i for i, u in enumerate(nodes)}
    seen: Set[Tuple[Any, ...]] = set()
    cycles: List[Tuple[Any, ...]] = []

    for start in tqdm(nodes, desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        stack = [(start, [start], {start})]
        while stack:
            u, path, used = stack.pop()
            for v in tqdm(G.neighbors(u), desc='Trapping-set analysis', leave=False, disable=expansion_contraction_config['print_level'] == 1):
                if v == start:
                    if len(path) >= 4 and len(path) <= max_len:
                        key = canonical_cycle(path)
                        if key not in seen:
                            seen.add(key)
                            cycles.append(key)
                            if max_cycles is not None and len(cycles) >= max_cycles:
                                return cycles
                elif v not in used:
                    if rank[v] < rank[start]:
                        continue
                    if len(path) + 1 <= max_len:
                        stack.append((v, path + [v], used | {v}))

    return cycles

def expand_from_cycle(
    G: nx.Graph,
    cycle: Iterable[Tuple[str, int]],
    depth: int=2,
) -> Set[Tuple[str, int]]:
    expanded = set(cycle)
    frontier_check = set(((typ, u) for typ, u in cycle if typ == 'c'))
    for _ in tqdm(range(depth), desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        nxt_checks = set()
        nei_vars = set()
        for c in tqdm(frontier_check, desc='Trapping-set analysis', leave=False, disable=expansion_contraction_config['print_level'] == 1):
            nei_var = [v for v in G.neighbors(c) if v not in expanded]
            nei_vars.update(nei_var)
            for v in nei_var:
                nxt_checks.update(G.neighbors(v))

        nxt_checks -= frontier_check
        nxt_checks -= expanded
        if not nxt_checks:
            break
        expanded |= nei_vars
        expanded |= nxt_checks
        frontier_check = nxt_checks

    return expanded

def contract_expanded_subgraph(
    matrices: Matrices,
    expanded_nodes: Iterable[Any],
    decoder_fails: DecoderFails,
    max_error_weight: int=4,
    seed_cycle: Optional[Tuple[Any, ...]]=None,
) -> Optional[TrappingSetRecord]:
    """
    Contraction step.

    1. Extract expanded subgraph G_EXP.
    2. Exhaustively corrupt variable nodes in G_EXP up to max_error_weight.
    3. Run decoder on each local error pattern.
    4. Union all supports that fail.
    5. Contract to induced TS subgraph on that union.
    """

    var_ids = tuple(sorted((j for typ, j in expanded_nodes if typ == 'v')))
    check_ids = tuple(sorted((i for typ, i in expanded_nodes if typ == 'c')))
    new_matrices = Matrices(H=matrices.H[list(check_ids), :][:, list(var_ids)], L=matrices.L[:][:, list(var_ids)], p=matrices.p[list(var_ids)])
    n_sub = len(var_ids)

    if n_sub == 0:
        return None
    max_w = min(max_error_weight, n_sub)
    failing_supports: Set[frozenset[int]] = set()

    for w in tqdm(range(1, max_w + 1), desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        for local_supp in tqdm(itertools.combinations(range(n_sub), w), desc='Trapping-set analysis', leave=False, disable=expansion_contraction_config['print_level'] == 1):
            e_sub = np.zeros(n_sub, dtype=np.uint8)
            e_sub[list(local_supp)] = 1
            if decoder_fails(new_matrices, e_sub):
                global_supp = frozenset((var_ids[k] for k in local_supp))
                failing_supports.add(global_supp)

    if not failing_supports:
        return None
    T = set()

    for supp in tqdm(failing_supports, desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        T |= set(supp)
    T = tuple(sorted(T))
    mu = min((len(supp) for supp in failing_supports))
    minimal_failures = tuple(sorted((tuple(sorted(supp)) for supp in failing_supports if len(supp) == mu)))
    deg = np.asarray(matrices.H[:, list(T)].sum(axis=1)).ravel().astype(int)
    checks = tuple((int(i) for i in np.flatnonzero(deg > 0)))
    odd_checks = tuple((int(i) for i in np.flatnonzero(deg % 2 == 1)))

    return TrappingSetRecord(
        variables=T,
        checks=checks,
        odd_checks=odd_checks,
        a=len(T),
        b=len(odd_checks),
        critical_number=mu,
        strength=len(minimal_failures),
        minimal_failures=minimal_failures,
        seed_cycle=tuple(seed_cycle) if seed_cycle is not None else tuple(),
        expanded_variables=var_ids,
        expanded_checks=check_ids,
    )

def expansion_contraction(
    dem: stim.DetectorErrorModel,
    decoder_fails: DecoderFails,
    max_cycle_len: int=8,
    expand_depth: int=2,
    max_error_weight: int=4,
    max_cycles: Optional[int]=None,
    deduplicate: bool=True,
) -> List[TrappingSetRecord]:
    """
    Full Sub-Graph Expansion-Contraction method.

    Parameters
    ----------
    dem:
        Detector error model.
    decoder_fails:
        Failure oracle.
    max_cycle_len:
        Enumerate seed cycles up to this Tanner-graph length.
    expand_depth:
        BFS expansion radius around each seed cycle.
    max_error_weight:
        Exhaustively test error patterns up to this weight.
        Use None only for very small expanded subgraphs.
    max_cycles:
        Optional cap on number of seed cycles.
    deduplicate:
        Merge identical contracted TSs found from different seed cycles.
    """

    demMatrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    matrices = Matrices(H=demMatrices.check_matrix.toarray(), L=demMatrices.observables_matrix.toarray(), p=demMatrices.priors)
    G = build_tanner_graph(matrices.H)
    cycles = enumerate_short_cycles(G, max_len=max_cycle_len, max_cycles=max_cycles)
    records: List[TrappingSetRecord] = []

    for cyc in tqdm(cycles, desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        expanded_nodes = expand_from_cycle(G, cyc, depth=expand_depth)
        rec = contract_expanded_subgraph(
            matrices=matrices,
            expanded_nodes=expanded_nodes,
            decoder_fails=decoder_fails,
            max_error_weight=max_error_weight,
            seed_cycle=cyc,
        )
        if rec is not None:
            records.append(rec)

    if not deduplicate:
        return sorted(records, key=lambda r: (r.critical_number, r.a, r.b))
    by_T: Dict[Tuple[int, ...], TrappingSetRecord] = {}

    for rec in tqdm(records, desc='Trapping-set analysis', leave=True, disable=expansion_contraction_config['print_level'] == 1):
        key = rec.variables
        if key not in by_T:
            by_T[key] = rec
            continue
        old = by_T[key]

        if rec.critical_number < old.critical_number:
            by_T[key] = rec
        elif rec.critical_number == old.critical_number:
            merged_min = tuple(sorted(set(old.minimal_failures) | set(rec.minimal_failures)))
            by_T[key] = replace(old, strength=len(merged_min), minimal_failures=merged_min)

    return sorted(by_T.values(), key=lambda r: (r.critical_number, r.a, r.b))


In [ ]:
expansion_contraction_builder = CircuitBuilder(expansion_contraction_ca_codes['CA_9_2_5'])
expansion_contraction_circ = expansion_contraction_builder.get_circuit_level_circuit(error_rate=ErrorRate(p=0.023101, p_meas=0.0), d_rounds=1)
expansion_contraction_dem = expansion_contraction_circ.detector_error_model()


In [ ]:
errs = stim.Circuit.shortest_graphlike_error(expansion_contraction_circ, ignore_ungraphlike_errors=False)
len(errs)


In [ ]:
errs = expansion_contraction_circ.search_for_undetectable_logical_errors(dont_explore_detection_event_sets_with_size_above=10, dont_explore_edges_with_degree_above=14, dont_explore_edges_increasing_symptom_degree=False)
len(errs)


### Generate and inspect candidate records

Apply the configured construction, inspect a representative record and display its Tanner neighborhood. The construction changes the represented event, so track its variables and checks explicitly.


In [ ]:
records = expansion_contraction(
    expansion_contraction_dem,
    decoder_fails=bposd_fails_but_tesseract_succeeds,
    max_cycle_len=6,
    expand_depth=2,
    max_error_weight=3,
    max_cycles=5000,
)

from experiment.utils.trapping_sets import save_expansion_records

record_table, structural_summary = save_expansion_records(records, trace_dir=expansion_contraction_data_paths.trace(), stats_dir=expansion_contraction_data_paths.stats())
structural_summary


In [ ]:
records


In [ ]:
expansion_contraction_r = records[-8]
expansion_contraction_r


In [ ]:
from utils.graph_struct import build_tanner_graph_from_dem
from utils.graph_vis import visualize_graph_plotly, visualize_det_on_tanner_graph


In [ ]:
expansion_contraction_g, expansion_contraction_ = build_tanner_graph_from_dem(expansion_contraction_dem)
num_err = expansion_contraction_dem.num_errors
num_det = expansion_contraction_dem.num_detectors


In [ ]:
save_plotly(visualize_det_on_tanner_graph(expansion_contraction_g, id2str(expansion_contraction_r.checks, num_det), other_events={('error', 'green', 'circle'): id2str(expansion_contraction_r.variables, num_err)}), expansion_contraction_data_paths.output('cell_0015_1_html'))


In [ ]:
expansion_contraction_r.minimal_failures


### Verify a generated failure example

Decode a recorded minimal-failure support with both references and compare syndrome and logical observables. Structural appearance alone does not determine whether a correction is logically correct.


In [ ]:
err = id2bits(expansion_contraction_r.minimal_failures[1], num_err)
(err, err.shape)


In [ ]:
compiled_b = bposd.compile_decoder_for_dem(dem=expansion_contraction_dem)
compiled_t = tess.compile_decoder_for_dem(dem=expansion_contraction_dem)


In [ ]:
syndrome = compiled_b.decoder.H @ err % 2
corr_b = compiled_b.decoder.decode_to_correction(syndrome)
corr_t = compiled_t.decoder.decode_to_errors(syndrome)
(corr_b, corr_t)


In [ ]:
compiled_b.decoder.L @ err % 2


In [ ]:
compiled_b.decoder.L @ corr_b % 2


In [ ]:
compiled_b.decoder.L @ id2bits(corr_t, num_err) % 2


In [ ]:
bits2id(syndrome)


In [ ]:
bits2id(corr_b)


In [ ]:
save_plotly(visualize_det_on_tanner_graph(expansion_contraction_g, bits2str(syndrome), other_events={
    ('origin', 'orange', 'circle'): bits2str(err),
    ('corr_b', 'green', 'circle'): bits2str(corr_b),
}), expansion_contraction_data_paths.output('cell_0025_1_html'))


## Different Error Rate

### Prepare the p-dependent construction sweeps

Read the named historical seed snapshot and prepare the sweep output paths. The next sections vary depth and maximum support weight with their own saved records.


In [ ]:
import pandas as pd

TS_DIR = expansion_contraction_data_paths.output('trapping_set')
TS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
expansion_contraction_max_shots = 1000000000
d_rounds = 1
noise_model = 'circuit_level'
expansion_contraction_cases_dir = expansion_contraction_data_paths.input('noise_model_d_rounds_perfect_meas_shots_expansion_contraction_max_shots', noise_model=noise_model, d_rounds=d_rounds, expansion_contraction_max_shots=expansion_contraction_max_shots)
assert expansion_contraction_cases_dir.is_dir(), expansion_contraction_cases_dir
expansion_contraction_types = {
    'detector_events': str,
    'error_events': str,
    'true_observables': str,
    'bposd_pred_observables': str,
    'tesseract_pred_observables': str,
}

expansion_contraction_bfts_cases = pd.read_csv(expansion_contraction_data_paths.input('bfts_cases_csv', noise_model=noise_model, d_rounds=d_rounds, expansion_contraction_max_shots=expansion_contraction_max_shots), dtype=expansion_contraction_types)
expansion_contraction_p_values = expansion_contraction_bfts_cases['p'].unique()
expansion_contraction_p_values


### Depth = 2, Weight = 4

### Depth 2 with maximum weight 4

Collect the configured p sweep, summarize record/probability fields and inspect structural hashes. Generated-event counts and probability-weighted summaries answer different questions.


In [ ]:
depth = 2
max_error_weight = 4
save_path = expansion_contraction_data_paths.output('ts_depth_depth__w_max_error_weight_csv', depth=depth, max_error_weight=max_error_weight)


In [ ]:
from experiment.utils.trapping_sets import collect_expansion_sweep

expansion_contraction_stats = expansion_contraction_data_paths.stats()
expansion_contraction_stats.mkdir(parents=True, exist_ok=True)
ts_data, expansion_contraction_dem = collect_expansion_sweep(
    expansion_contraction_ca_codes['CA_9_2_5'],
    expansion_contraction_p_values,
    expansion_contraction=expansion_contraction,
    decoder_fails=bposd_fails_but_tesseract_succeeds,
    depth=depth,
    max_error_weight=max_error_weight,
    trace_path=save_path,
    stats_path=expansion_contraction_data_paths.output('parameter_1__summary_csv', parameter_1=save_path.stem),
    print_level=expansion_contraction_config['print_level'],
)


In [ ]:
ts_data = pd.read_csv(save_path)


In [ ]:
ts_data


In [ ]:
ts_data[80:100]


In [ ]:
summary_by_p = ts_data.groupby('p').agg(
    num_records=('p', 'size'),
    num_unique_ts=('variables', 'nunique'),
    mean_a=('a', 'mean'),
    mean_b=('b', 'mean'),
    min_critical=('critical_number', 'min'),
    mean_critical=('critical_number', 'mean'),
    max_strength=('strength', 'max'),
    mean_strength=('strength', 'mean'),
).reset_index()
expansion_contraction_stats = expansion_contraction_data_paths.stats()
expansion_contraction_stats.mkdir(parents=True, exist_ok=True)

summary_by_p.to_csv(expansion_contraction_data_paths.output('parameter_1__summary_csv', parameter_1=save_path.stem), index=False)
summary_by_p


In [ ]:
expansion_contraction_g, expansion_contraction_ = build_tanner_graph_from_dem(expansion_contraction_dem)


In [ ]:
ts_data['WL_Hash'] = ts_data.apply(lambda row: nx.weisfeiler_lehman_graph_hash(expansion_contraction_g.subgraph(list(parse_node_ids(row['checks'])) + list(map(lambda x: x + num_det, list(parse_node_ids(row['variables']))))), node_attr='kind'), axis=1)
ts_data


In [ ]:
ts_data.count()


### Depth = 2, Weight = 3

### Depth 2 with maximum weight 3

Repeat the construction at the smaller weight bound and inspect selected neighborhoods. Compare with the previous sweep while retaining the changed construction bound and denominator.


In [ ]:
depth = 2
max_error_weight = 3
save_path = expansion_contraction_data_paths.output('ts_depth_depth__w_max_error_weight_csv', depth=depth, max_error_weight=max_error_weight)


In [ ]:
from experiment.utils.trapping_sets import collect_expansion_sweep

expansion_contraction_stats = expansion_contraction_data_paths.stats()
expansion_contraction_stats.mkdir(parents=True, exist_ok=True)
ts_data, expansion_contraction_dem = collect_expansion_sweep(
    expansion_contraction_ca_codes['CA_9_2_5'],
    expansion_contraction_p_values,
    expansion_contraction=expansion_contraction,
    decoder_fails=bposd_fails_but_tesseract_succeeds,
    depth=depth,
    max_error_weight=max_error_weight,
    trace_path=save_path,
    stats_path=expansion_contraction_data_paths.output('parameter_1__summary_csv', parameter_1=save_path.stem),
    print_level=expansion_contraction_config['print_level'],
)


In [ ]:
ts_data = pd.read_csv(save_path)


In [ ]:
ts_data


In [ ]:
summary_by_p = ts_data.groupby('p').agg(
    num_records=('p', 'size'),
    num_unique_ts=('variables', 'nunique'),
    mean_a=('a', 'mean'),
    mean_b=('b', 'mean'),
    min_critical=('critical_number', 'min'),
    mean_critical=('critical_number', 'mean'),
    max_strength=('strength', 'max'),
    mean_strength=('strength', 'mean'),
).reset_index()
expansion_contraction_stats = expansion_contraction_data_paths.stats()
expansion_contraction_stats.mkdir(parents=True, exist_ok=True)

summary_by_p.to_csv(expansion_contraction_data_paths.output('parameter_1__summary_csv', parameter_1=save_path.stem), index=False)
summary_by_p


In [ ]:
expansion_contraction_g, expansion_contraction_ = build_tanner_graph_from_dem(expansion_contraction_dem)


In [ ]:
ts_data['WL_Hash'] = ts_data.apply(lambda row: nx.weisfeiler_lehman_graph_hash(expansion_contraction_g.subgraph(list(parse_node_ids(row['checks'])) + list(map(lambda x: x + num_det, list(parse_node_ids(row['variables']))))), node_attr='kind'), axis=1)
ts_data


In [ ]:
ts_data.count()


In [ ]:
expansion_contraction_row = ts_data.iloc[82]
save_plotly(visualize_det_on_tanner_graph(expansion_contraction_g, id2str(parse_node_ids(expansion_contraction_row['checks']), num_det), other_events={
    ('error', 'green', 'circle'): id2str(parse_node_ids(expansion_contraction_row['variables']), num_err),
}), expansion_contraction_data_paths.output('cell_0050_1_html'))


In [ ]:
nodes = list(parse_node_ids(expansion_contraction_row['checks'])) + list(map(lambda x: x + num_det, list(parse_node_ids(expansion_contraction_row['variables']))))
save_plotly(visualize_graph_plotly(flatten_graph(expansion_contraction_g.subgraph(nodes))), expansion_contraction_data_paths.output('cell_0051_1_html'))


In [ ]:
expansion_contraction_row = ts_data.iloc[90]
save_plotly(visualize_det_on_tanner_graph(expansion_contraction_g, id2str(parse_node_ids(expansion_contraction_row['checks']), num_det), other_events={
    ('error', 'green', 'circle'): id2str(parse_node_ids(expansion_contraction_row['variables']), num_err),
}), expansion_contraction_data_paths.output('cell_0052_1_html'))


In [ ]:
nodes = list(parse_node_ids(expansion_contraction_row['checks'])) + list(map(lambda x: x + num_det, list(parse_node_ids(expansion_contraction_row['variables']))))
save_plotly(visualize_graph_plotly(flatten_graph(expansion_contraction_g.subgraph(nodes))), expansion_contraction_data_paths.output('cell_0053_1_html'))


### Depth = 2, Weight = 2

### Depth 2 with maximum weight 2

Repeat at weight bound 2, then display representative supports. A changed count may reflect the changed available event set rather than a decoder improvement.


In [ ]:
depth = 2
max_error_weight = 2
save_path = expansion_contraction_data_paths.output('ts_depth_depth__w_max_error_weight_csv', depth=depth, max_error_weight=max_error_weight)


In [ ]:
from experiment.utils.trapping_sets import collect_expansion_sweep

expansion_contraction_stats = expansion_contraction_data_paths.stats()
expansion_contraction_stats.mkdir(parents=True, exist_ok=True)
ts_data, expansion_contraction_dem = collect_expansion_sweep(
    expansion_contraction_ca_codes['CA_9_2_5'],
    expansion_contraction_p_values,
    expansion_contraction=expansion_contraction,
    decoder_fails=bposd_fails_but_tesseract_succeeds,
    depth=depth,
    max_error_weight=max_error_weight,
    trace_path=save_path,
    stats_path=expansion_contraction_data_paths.output('parameter_1__summary_csv', parameter_1=save_path.stem),
    print_level=expansion_contraction_config['print_level'],
)


In [ ]:
ts_data = pd.read_csv(save_path)
summary_by_p = ts_data.groupby('p').agg(
    num_records=('p', 'size'),
    num_unique_ts=('variables', 'nunique'),
    mean_a=('a', 'mean'),
    mean_b=('b', 'mean'),
    min_critical=('critical_number', 'min'),
    mean_critical=('critical_number', 'mean'),
    max_strength=('strength', 'max'),
    mean_strength=('strength', 'mean'),
).reset_index()
expansion_contraction_stats = expansion_contraction_data_paths.stats()
expansion_contraction_stats.mkdir(parents=True, exist_ok=True)

summary_by_p.to_csv(expansion_contraction_data_paths.output('parameter_1__summary_csv', parameter_1=save_path.stem), index=False)


In [ ]:
ts_data


In [ ]:
expansion_contraction_g, expansion_contraction_ = build_tanner_graph_from_dem(expansion_contraction_dem)
num_err = expansion_contraction_dem.num_errors
num_det = expansion_contraction_dem.num_detectors


In [ ]:
ts_data['WL_Hash'] = ts_data.apply(lambda row: nx.weisfeiler_lehman_graph_hash(expansion_contraction_g.subgraph(list(parse_node_ids(row['checks'])) + list(map(lambda x: x + num_det, list(parse_node_ids(row['variables']))))), node_attr='kind'), axis=1)
ts_data


In [ ]:
expansion_contraction_row = ts_data.iloc[-1]
save_plotly(visualize_det_on_tanner_graph(expansion_contraction_g, id2str(parse_node_ids(expansion_contraction_row['checks']), num_det), other_events={
    ('error', 'green', 'circle'): id2str(parse_node_ids(expansion_contraction_row['variables']), num_err),
}), expansion_contraction_data_paths.output('cell_0062_1_html'))


In [ ]:
nodes = list(parse_node_ids(expansion_contraction_row['checks'])) + list(map(lambda x: x + num_det, list(parse_node_ids(expansion_contraction_row['variables']))))
save_plotly(visualize_graph_plotly(flatten_graph(expansion_contraction_g.subgraph(nodes))), expansion_contraction_data_paths.output('cell_0063_1_html'))


### Depth = 1, Weight = 4

### Depth 1 with maximum weight 4

Reduce neighborhood depth while restoring the weight bound to 4. This provides a separate construction comparison, not a continuation of the weight-2 population.


In [ ]:
depth = 1
max_error_weight = 4


In [ ]:
save_path = expansion_contraction_data_paths.output('ts_depth_depth__w_max_error_weight_csv', depth=depth, max_error_weight=max_error_weight)

from experiment.utils.trapping_sets import collect_expansion_sweep

expansion_contraction_stats = expansion_contraction_data_paths.stats()
expansion_contraction_stats.mkdir(parents=True, exist_ok=True)
ts_data, expansion_contraction_dem = collect_expansion_sweep(
    expansion_contraction_ca_codes['CA_9_2_5'],
    expansion_contraction_p_values,
    expansion_contraction=expansion_contraction,
    decoder_fails=bposd_fails_but_tesseract_succeeds,
    depth=depth,
    max_error_weight=max_error_weight,
    trace_path=save_path,
    stats_path=expansion_contraction_data_paths.output('parameter_1__summary_csv', parameter_1=save_path.stem),
    print_level=expansion_contraction_config['print_level'],
)
